In [7]:
import matplotlib.pyplot as plt

# You may change the mhealth_activity module but your algorithm must support the original version
from mhealth_activity import Recording, Trace, Activity, WatchLocation, Path

# For interactive plots, uncomment the following line
# %matplotlib widget

In [15]:
def prefix_slice(x, fraction=0.30):
    if len(x) < 10:
        return np.array([], dtype=float)
    n = max(10, int(len(x) * fraction))
    return x[:n]


def early_route_features(signal, prefix):
    feats = {}

    early = prefix_slice(signal, fraction=0.30)

    if len(early) < 10:
        return {
            f"{prefix}_early_mean": np.nan,
            f"{prefix}_early_std": np.nan,
            f"{prefix}_early_range": np.nan,
            f"{prefix}_early_net_change": np.nan,
            f"{prefix}_early_total_abs_change": np.nan,
        }

    diff = np.diff(early)

    return {
        f"{prefix}_early_mean": np.mean(early),
        f"{prefix}_early_std": np.std(early),
        f"{prefix}_early_range": np.max(early) - np.min(early),
        f"{prefix}_early_net_change": early[-1] - early[0],
        f"{prefix}_early_total_abs_change": np.sum(np.abs(diff)),
    }

In [17]:
import os
import numpy as np
import pandas as pd

from scipy.signal import find_peaks
from scipy.stats import skew, kurtosis

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier

import joblib


# ============================================================
# Trace helpers
# ============================================================

def trace_values(recording, key):
    if key not in recording.data:
        return np.array([], dtype=float)

    tr = recording.data[key]

    for attr in ["values", "data", "y", "v"]:
        if hasattr(tr, attr):
            arr = np.asarray(getattr(tr, attr), dtype=float)
            return arr[np.isfinite(arr)]

    try:
        arr = np.asarray(tr, dtype=float)
        return arr[np.isfinite(arr)]
    except Exception:
        raise ValueError(f"Could not extract numeric values from {key}")


def trace_total_time(recording, key="timestamp"):
    if key not in recording.data:
        return np.nan
    return getattr(recording.data[key], "total_time", np.nan)


def magnitude(x, y, z):
    n = min(len(x), len(y), len(z))
    if n == 0:
        return np.array([], dtype=float)
    return np.sqrt(x[:n] ** 2 + y[:n] ** 2 + z[:n] ** 2)


# ============================================================
# Generic feature helpers
# ============================================================

def basic_stats(x, prefix):
    if len(x) < 5:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_max": np.nan,
            f"{prefix}_range": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_iqr": np.nan,
            f"{prefix}_skew": np.nan,
            f"{prefix}_kurt": np.nan,
        }

    return {
        f"{prefix}_mean": np.mean(x),
        f"{prefix}_std": np.std(x),
        f"{prefix}_min": np.min(x),
        f"{prefix}_max": np.max(x),
        f"{prefix}_range": np.max(x) - np.min(x),
        f"{prefix}_median": np.median(x),
        f"{prefix}_iqr": np.percentile(x, 75) - np.percentile(x, 25),
        f"{prefix}_skew": skew(x),
        f"{prefix}_kurt": kurtosis(x),
    }


def peak_features(x, prefix):
    if len(x) < 20:
        return {
            f"{prefix}_n_peaks": np.nan,
            f"{prefix}_peak_rate": np.nan,
            f"{prefix}_peak_prom_mean": np.nan,
        }

    x_centered = x - np.median(x)
    prom = max(np.std(x_centered), 1e-6)

    peaks, props = find_peaks(
        x_centered,
        prominence=prom,
        distance=5
    )

    if len(peaks) == 0:
        return {
            f"{prefix}_n_peaks": 0,
            f"{prefix}_peak_rate": 0,
            f"{prefix}_peak_prom_mean": 0,
        }

    return {
        f"{prefix}_n_peaks": len(peaks),
        f"{prefix}_peak_rate": len(peaks) / len(x),
        f"{prefix}_peak_prom_mean": np.mean(props["prominences"]),
    }


def segment_stats(x, prefix, n_segments=8):
    feats = {}

    for i in range(n_segments):
        feats[f"{prefix}_seg{i}_mean"] = np.nan
        feats[f"{prefix}_seg{i}_std"] = np.nan
        feats[f"{prefix}_seg{i}_range"] = np.nan
        feats[f"{prefix}_seg{i}_median"] = np.nan

    if len(x) < n_segments * 5:
        return feats

    segments = np.array_split(x, n_segments)

    for i, seg in enumerate(segments):
        feats[f"{prefix}_seg{i}_mean"] = np.mean(seg)
        feats[f"{prefix}_seg{i}_std"] = np.std(seg)
        feats[f"{prefix}_seg{i}_range"] = np.max(seg) - np.min(seg)
        feats[f"{prefix}_seg{i}_median"] = np.median(seg)

    return feats


def burst_features(x, prefix):
    if len(x) < 20:
        return {
            f"{prefix}_burst_count": np.nan,
            f"{prefix}_burst_ratio": np.nan,
            f"{prefix}_burst_mean_value": np.nan,
        }

    threshold = np.percentile(x, 90)
    mask = x > threshold

    return {
        f"{prefix}_burst_count": np.sum(np.diff(mask.astype(int)) == 1),
        f"{prefix}_burst_ratio": np.mean(mask),
        f"{prefix}_burst_mean_value": np.mean(x[mask]) if np.any(mask) else 0,
    }


# ============================================================
# Heading / turn helpers
# ============================================================

def heading_array(x_mag, y_mag):
    n = min(len(x_mag), len(y_mag))
    if n < 10:
        return np.array([], dtype=float)

    return np.unwrap(np.arctan2(y_mag[:n], x_mag[:n]))


def heading_features(heading, prefix):
    if len(heading) < 10:
        return {
            f"{prefix}_total_abs_change": np.nan,
            f"{prefix}_net_change": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_start": np.nan,
            f"{prefix}_end": np.nan,
        }

    diff = np.diff(heading)

    return {
        f"{prefix}_total_abs_change": np.sum(np.abs(diff)),
        f"{prefix}_net_change": heading[-1] - heading[0],
        f"{prefix}_std": np.std(heading),
        f"{prefix}_start": heading[0],
        f"{prefix}_end": heading[-1],
    }


def segment_net_change(x, prefix, n_segments=8):
    feats = {}

    for i in range(n_segments):
        feats[f"{prefix}_seg{i}_net_change"] = np.nan
        feats[f"{prefix}_seg{i}_total_abs_change"] = np.nan
        feats[f"{prefix}_seg{i}_signed_pos_sum"] = np.nan
        feats[f"{prefix}_seg{i}_signed_neg_sum"] = np.nan

    if len(x) < n_segments * 5:
        return feats

    segments = np.array_split(x, n_segments)

    for i, seg in enumerate(segments):
        diff = np.diff(seg)

        feats[f"{prefix}_seg{i}_net_change"] = seg[-1] - seg[0]
        feats[f"{prefix}_seg{i}_total_abs_change"] = np.sum(np.abs(diff))
        feats[f"{prefix}_seg{i}_signed_pos_sum"] = np.sum(diff[diff > 0])
        feats[f"{prefix}_seg{i}_signed_neg_sum"] = np.sum(diff[diff < 0])

    return feats


def signed_turn_features(signal, prefix, n_segments=8):
    """
    Extract signed turning/change features.

    Works for:
    - heading proxy differences
    - gyro axis signals

    Positive/negative does not always literally mean left/right,
    but the signed pattern can help classify routes.
    """
    feats = {}

    default_keys = [
        "pos_turn_count",
        "neg_turn_count",
        "pos_turn_sum",
        "neg_turn_sum",
        "turn_balance",
        "abs_turn_sum",
        "large_turn_count",
    ]

    for k in default_keys:
        feats[f"{prefix}_{k}"] = np.nan

    for i in range(n_segments):
        feats[f"{prefix}_seg{i}_pos_turn_count"] = np.nan
        feats[f"{prefix}_seg{i}_neg_turn_count"] = np.nan
        feats[f"{prefix}_seg{i}_pos_turn_sum"] = np.nan
        feats[f"{prefix}_seg{i}_neg_turn_sum"] = np.nan
        feats[f"{prefix}_seg{i}_turn_balance"] = np.nan

    if len(signal) < 20:
        return feats

    diff = np.diff(signal)

    abs_diff = np.abs(diff)
    threshold = np.percentile(abs_diff, 90)

    pos = diff > threshold
    neg = diff < -threshold
    large = abs_diff > threshold

    pos_sum = np.sum(diff[pos]) if np.any(pos) else 0
    neg_sum = np.sum(diff[neg]) if np.any(neg) else 0

    feats[f"{prefix}_pos_turn_count"] = np.sum(pos)
    feats[f"{prefix}_neg_turn_count"] = np.sum(neg)
    feats[f"{prefix}_pos_turn_sum"] = pos_sum
    feats[f"{prefix}_neg_turn_sum"] = neg_sum
    feats[f"{prefix}_turn_balance"] = pos_sum + neg_sum
    feats[f"{prefix}_abs_turn_sum"] = np.sum(abs_diff)
    feats[f"{prefix}_large_turn_count"] = np.sum(large)

    segments = np.array_split(signal, n_segments)

    for i, seg in enumerate(segments):
        if len(seg) < 5:
            continue

        d = np.diff(seg)
        abs_d = np.abs(d)
        th = np.percentile(abs_d, 90)

        p = d > th
        n = d < -th

        p_sum = np.sum(d[p]) if np.any(p) else 0
        n_sum = np.sum(d[n]) if np.any(n) else 0

        feats[f"{prefix}_seg{i}_pos_turn_count"] = np.sum(p)
        feats[f"{prefix}_seg{i}_neg_turn_count"] = np.sum(n)
        feats[f"{prefix}_seg{i}_pos_turn_sum"] = p_sum
        feats[f"{prefix}_seg{i}_neg_turn_sum"] = n_sum
        feats[f"{prefix}_seg{i}_turn_balance"] = p_sum + n_sum

    return feats


def gyro_signed_features(axis_signal, prefix, n_segments=8):
    """
    Signed gyro features.
    These capture directional rotation bursts per axis.
    """
    feats = {}

    for key in [
        "positive_energy",
        "negative_energy",
        "signed_energy_balance",
        "positive_ratio",
        "negative_ratio",
        "large_positive_count",
        "large_negative_count",
    ]:
        feats[f"{prefix}_{key}"] = np.nan

    for i in range(n_segments):
        feats[f"{prefix}_seg{i}_signed_sum"] = np.nan
        feats[f"{prefix}_seg{i}_abs_sum"] = np.nan
        feats[f"{prefix}_seg{i}_positive_ratio"] = np.nan
        feats[f"{prefix}_seg{i}_negative_ratio"] = np.nan

    if len(axis_signal) < 20:
        return feats

    x = axis_signal

    pos = x[x > 0]
    neg = x[x < 0]

    feats[f"{prefix}_positive_energy"] = np.mean(pos ** 2) if len(pos) else 0
    feats[f"{prefix}_negative_energy"] = np.mean(neg ** 2) if len(neg) else 0
    feats[f"{prefix}_signed_energy_balance"] = (
        feats[f"{prefix}_positive_energy"] - feats[f"{prefix}_negative_energy"]
    )
    feats[f"{prefix}_positive_ratio"] = np.mean(x > 0)
    feats[f"{prefix}_negative_ratio"] = np.mean(x < 0)

    threshold = np.percentile(np.abs(x), 90)
    feats[f"{prefix}_large_positive_count"] = np.sum(x > threshold)
    feats[f"{prefix}_large_negative_count"] = np.sum(x < -threshold)

    segments = np.array_split(x, n_segments)

    for i, seg in enumerate(segments):
        feats[f"{prefix}_seg{i}_signed_sum"] = np.sum(seg)
        feats[f"{prefix}_seg{i}_abs_sum"] = np.sum(np.abs(seg))
        feats[f"{prefix}_seg{i}_positive_ratio"] = np.mean(seg > 0)
        feats[f"{prefix}_seg{i}_negative_ratio"] = np.mean(seg < 0)

    return feats


# ============================================================
# Feature extraction
# ============================================================

def extract_path_features(recording, n_segments=8):
    features = {}

    features["duration_sec"] = trace_total_time(recording, "timestamp")

    # --------------------------------------------------------
    # Altitude / uphill feature
    # --------------------------------------------------------
    altitude = trace_values(recording, "altitude")

    if len(altitude) >= 10:
        initial_alt = np.median(altitude[:10])
        final_alt = np.median(altitude[-10:])
        alt_delta = final_alt - initial_alt

        features["alt_initial"] = initial_alt
        features["alt_final"] = final_alt
        features["alt_delta"] = alt_delta
        features["is_uphill"] = int(alt_delta > 0)
    else:
        features["alt_initial"] = np.nan
        features["alt_final"] = np.nan
        features["alt_delta"] = np.nan
        features["is_uphill"] = np.nan

    features.update(segment_stats(altitude, "altitude", n_segments=n_segments))

    # --------------------------------------------------------
    # Watch gyroscope
    # --------------------------------------------------------
    gx = trace_values(recording, "gx")
    gy = trace_values(recording, "gy")
    gz = trace_values(recording, "gz")

    gyro_mag = magnitude(gx, gy, gz)

    features.update(basic_stats(gyro_mag, "gyro_mag"))
    features.update(peak_features(gyro_mag, "gyro_mag"))
    features.update(burst_features(gyro_mag, "gyro_mag"))
    features.update(segment_stats(gyro_mag, "gyro_mag", n_segments=n_segments))

    for axis_name, axis_signal in [("gx", gx), ("gy", gy), ("gz", gz)]:
        features.update(basic_stats(axis_signal, axis_name))
        features.update(segment_stats(axis_signal, axis_name, n_segments=n_segments))
        features.update(gyro_signed_features(axis_signal, axis_name, n_segments=n_segments))

    # --------------------------------------------------------
    # Phone gyroscope
    # --------------------------------------------------------
    pgx = trace_values(recording, "phone_gx")
    pgy = trace_values(recording, "phone_gy")
    pgz = trace_values(recording, "phone_gz")

    phone_gyro_mag = magnitude(pgx, pgy, pgz)

    features.update(basic_stats(phone_gyro_mag, "phone_gyro_mag"))
    features.update(peak_features(phone_gyro_mag, "phone_gyro_mag"))
    features.update(burst_features(phone_gyro_mag, "phone_gyro_mag"))
    features.update(segment_stats(phone_gyro_mag, "phone_gyro_mag", n_segments=n_segments))

    for axis_name, axis_signal in [
        ("phone_gx", pgx),
        ("phone_gy", pgy),
        ("phone_gz", pgz)
    ]:
        features.update(basic_stats(axis_signal, axis_name))
        features.update(segment_stats(axis_signal, axis_name, n_segments=n_segments))
        features.update(gyro_signed_features(axis_signal, axis_name, n_segments=n_segments))

    # --------------------------------------------------------
    # Watch magnetometer + heading proxy
    # --------------------------------------------------------
    mx = trace_values(recording, "mx")
    my = trace_values(recording, "my")
    mz = trace_values(recording, "mz")

    mag_mag = magnitude(mx, my, mz)
    watch_heading = heading_array(mx, my)

    features.update(basic_stats(mag_mag, "watch_mag_mag"))
    features.update(segment_stats(mag_mag, "watch_mag_mag", n_segments=n_segments))

    for axis_name, axis_signal in [("mx", mx), ("my", my), ("mz", mz)]:
        features.update(basic_stats(axis_signal, axis_name))
        features.update(segment_stats(axis_signal, axis_name, n_segments=n_segments))

    features.update(heading_features(watch_heading, "watch_heading"))
    features.update(segment_stats(watch_heading, "watch_heading", n_segments=n_segments))
    features.update(segment_net_change(watch_heading, "watch_heading", n_segments=n_segments))
    features.update(signed_turn_features(watch_heading, "watch_heading_turn", n_segments=n_segments))

    # --------------------------------------------------------
    # Phone magnetometer + heading proxy
    # --------------------------------------------------------
    phone_mx = trace_values(recording, "phone_mx")
    phone_my = trace_values(recording, "phone_my")
    phone_mz = trace_values(recording, "phone_mz")

    phone_mag_mag = magnitude(phone_mx, phone_my, phone_mz)
    phone_heading = heading_array(phone_mx, phone_my)

    features.update(basic_stats(phone_mag_mag, "phone_mag_mag"))
    features.update(segment_stats(phone_mag_mag, "phone_mag_mag", n_segments=n_segments))

    for axis_name, axis_signal in [
        ("phone_mx", phone_mx),
        ("phone_my", phone_my),
        ("phone_mz", phone_mz)
    ]:
        features.update(basic_stats(axis_signal, axis_name))
        features.update(segment_stats(axis_signal, axis_name, n_segments=n_segments))

    features.update(heading_features(phone_heading, "phone_heading"))
    features.update(segment_stats(phone_heading, "phone_heading", n_segments=n_segments))
    features.update(segment_net_change(phone_heading, "phone_heading", n_segments=n_segments))
    features.update(signed_turn_features(phone_heading, "phone_heading_turn", n_segments=n_segments))

    return features


# ============================================================
# Two-stage classifier
# ============================================================

class TwoStagePathClassifier:
    def __init__(self):
        self.direction_model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", ExtraTreesClassifier(
                n_estimators=500,
                random_state=1,
                class_weight="balanced",
                n_jobs=-1
            ))
        ])

        self.uphill_model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", ExtraTreesClassifier(
                n_estimators=900,
                random_state=2,
                class_weight="balanced",
                n_jobs=-1,
                max_features="sqrt"
            ))
        ])

        self.downhill_model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", ExtraTreesClassifier(
                n_estimators=700,
                random_state=3,
                class_weight="balanced",
                n_jobs=-1,
                max_features="sqrt"
            ))
        ])

        self.feature_columns = None

    def fit(self, X, y):
        self.feature_columns = list(X.columns)

        y_direction = np.isin(y, [0, 1, 2]).astype(int)
        self.direction_model.fit(X, y_direction)

        uphill_mask = np.isin(y, [0, 1, 2])
        downhill_mask = np.isin(y, [3, 4])

        self.uphill_model.fit(X[uphill_mask], y[uphill_mask])
        self.downhill_model.fit(X[downhill_mask], y[downhill_mask])

        return self

    def predict(self, X):
        X = X[self.feature_columns]

        direction_pred = self.direction_model.predict(X)
        preds = []

        for i in range(len(X)):
            row = X.iloc[[i]]

            if direction_pred[i] == 1:
                preds.append(self.uphill_model.predict(row)[0])
            else:
                preds.append(self.downhill_model.predict(row)[0])

        return np.array(preds)


# ============================================================
# Load recordings
# ============================================================

TRAIN_DIR = "data/train"

recordings = []
path_labels = []
filenames = []

for filename in sorted(os.listdir(TRAIN_DIR)):
    if not filename.endswith(".pkl"):
        continue

    filepath = os.path.join(TRAIN_DIR, filename)
    rec = Recording(filepath)

    if rec.labels is None:
        continue

    recordings.append(rec)
    path_labels.append(rec.labels["path_idx"])
    filenames.append(filename)

y = np.array(path_labels)

print("Loaded recordings:", len(recordings))
print("\nPath label counts:")
print(pd.Series(y).value_counts().sort_index())


# ============================================================
# Extract features
# ============================================================

feature_rows = []

for rec in recordings:
    feature_rows.append(extract_path_features(rec, n_segments=8))

X = pd.DataFrame(feature_rows)

print("\nFeature matrix shape:", X.shape)
print("Number of features:", len(X.columns))


# ============================================================
# Debug uphill feature
# ============================================================

debug_df = X.copy()
debug_df["path_idx"] = y
debug_df["filename"] = filenames

print("\nAltitude direction by path:")
print(debug_df.groupby("path_idx")[["alt_delta", "is_uphill"]].mean())


# ============================================================
# Train / validation split
# ============================================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================================
# Train model
# ============================================================

model = TwoStagePathClassifier()
model.fit(X_train, y_train)


# ============================================================
# Evaluate full model
# ============================================================

y_pred = model.predict(X_val)

print("\nFull path validation accuracy:")
print(accuracy_score(y_val, y_pred))

print("\nFull path classification report:")
print(classification_report(y_val, y_pred))

print("\nFull path confusion matrix:")
print(confusion_matrix(y_val, y_pred))


# ============================================================
# Evaluate uphill/downhill
# ============================================================

y_direction_true = np.isin(y_val, [0, 1, 2]).astype(int)
y_direction_pred = model.direction_model.predict(X_val)

print("\nUphill/downhill accuracy:")
print(accuracy_score(y_direction_true, y_direction_pred))

print("\nUphill/downhill confusion matrix:")
print(confusion_matrix(y_direction_true, y_direction_pred))


# ============================================================
# Evaluate within groups
# ============================================================

uphill_val_mask = np.isin(y_val, [0, 1, 2])
downhill_val_mask = np.isin(y_val, [3, 4])

if uphill_val_mask.sum() > 0:
    uphill_pred = model.uphill_model.predict(X_val[uphill_val_mask])
    print("\nUphill-only accuracy, paths 0/1/2:")
    print(accuracy_score(y_val[uphill_val_mask], uphill_pred))
    print(confusion_matrix(y_val[uphill_val_mask], uphill_pred))

if downhill_val_mask.sum() > 0:
    downhill_pred = model.downhill_model.predict(X_val[downhill_val_mask])
    print("\nDownhill-only accuracy, paths 3/4:")
    print(accuracy_score(y_val[downhill_val_mask], downhill_pred))
    print(confusion_matrix(y_val[downhill_val_mask], downhill_pred))


# ============================================================
# Feature importances
# ============================================================

uphill_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.uphill_model.named_steps["clf"].feature_importances_
}).sort_values("importance", ascending=False)

downhill_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.downhill_model.named_steps["clf"].feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 30 uphill model features:")
print(uphill_importance.head(30))

print("\nTop 30 downhill model features:")
print(downhill_importance.head(30))


# ============================================================
# Retrain on all data and save
# ============================================================

final_model = TwoStagePathClassifier()
final_model.fit(X, y)

joblib.dump(final_model, "groupXX_path_model.joblib")

print("\nSaved final model as groupXX_path_model.joblib")


# ============================================================
# Predict one recording
# ============================================================

# test_rec = Recording("data/test/test_trace_001.pkl")
# X_one = pd.DataFrame([extract_path_features(test_rec, n_segments=8)])
# predicted_path = int(final_model.predict(X_one)[0])
# print("Predicted path:", predicted_path)

Loaded recordings: 396

Path label counts:
0    82
1    76
2    74
3    71
4    93
Name: count, dtype: int64

Feature matrix shape: (396, 1171)
Number of features: 1171

Altitude direction by path:
          alt_delta  is_uphill
path_idx                      
0         42.070909   0.951220
1         40.432325   0.960526
2         49.026660   0.986486
3        -51.295879   0.014085
4        -44.035477   0.043011

Full path validation accuracy:
0.7375

Full path classification report:
              precision    recall  f1-score   support

           0       0.50      0.35      0.41        17
           1       0.82      0.93      0.88        15
           2       0.44      0.53      0.48        15
           3       1.00      0.86      0.92        14
           4       0.90      1.00      0.95        19

    accuracy                           0.74        80
   macro avg       0.73      0.74      0.73        80
weighted avg       0.73      0.74      0.73        80


Full path confusion ma